# Network Egress, SSRF, and External Resource Security

Build and evaluate a brokered outbound-fetch boundary. The notebook is deterministic: it performs no real DNS lookup, HTTP request, model call, or host-network fallback.

## Objectives and safety contract

You will expose only a URL proposal; derive identity, tenant, method, policy, DNS, and budgets from trusted state; bind approved DNS answers to the actual connection; re-authorize redirects; constrain responses; and measure both attack containment and useful valid requests. Synthetic public addresses and elapsed times are teaching fixtures, not production benchmarks.

In [ ]:
from dataclasses import asdict
import importlib.util
from pathlib import Path
import sys

def find_course():
    suffix = Path('curriculum/roadmap/intermediate/06-network-egress-ssrf-and-external-resource-security')
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / suffix
        if (candidate / 'lab.py').exists():
            return candidate
    if (Path.cwd() / 'lab.py').exists():
        return Path.cwd()
    raise FileNotFoundError('course directory not found')

COURSE = find_course()
def load_module(name, filename):
    spec = importlib.util.spec_from_file_location(name, COURSE / filename)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

lab = load_module('course06_lab_notebook', 'lab.py')
COURSE.name

## Trust boundary

![Brokered external fetch boundary](architecture.svg)

The URL, DNS answer, redirect, peer, and response are untrusted. Only authenticated application state and tenant-owned policy confer authority.

In [ ]:
scenario = lab.build_scenario()
policy = scenario.policies.policies['north']
{
    'hosts': sorted(policy.allow_hosts),
    'paths': policy.path_prefixes,
    'methods': sorted(policy.allowed_methods),
    'redirects': policy.max_redirects,
    'wire_bytes': policy.max_bytes,
    'decoded_bytes': policy.max_decoded_bytes,
    'deadline_seconds': policy.timeout_seconds,
}

## 1. Break the host-only baseline

A hostname allowlist cannot see private DNS answers, connection peers, redirects, response expansion, or replay. The intentionally unsafe helper exists only to make that gap observable.

In [ ]:
proposal = 'https://research.example.test/docs/guide'
scenario = lab.build_scenario()
scenario.resolver.records['research.example.test'] = ('169.254.169.254',)
baseline = lab.unsafe_host_only_baseline(proposal, policy.allow_hosts)
controlled = scenario.application.fetch(attestation_id='attest:north', url=proposal, request_id='nb:private', now=lab.NOW)
{'unsafe_baseline_allows': baseline, 'boundary': controlled.status.value, 'reason': controlled.reason, 'network_calls': scenario.transport.dispatches}

## 2. Canonicalize once and reject ambiguity

The parser rejects insecure schemes, user information, raw IPs, fragments, dot segments, nested encoding, and non-standard ports instead of trying to repair them.

In [ ]:
samples = [
    'https://Research.Example.Test./docs/guide?edition=current',
    'http://research.example.test/docs/guide',
    'https://user@research.example.test/docs/guide',
    'https://169.254.169.254/latest/meta-data',
    'https://research.example.test/docs/%252e%252e/admin',
]
[(url, result.normalized if result else None, reason) for url in samples for result, reason in [lab.canonicalize_url(url)]]

## 3. Validate every A and AAAA answer

One unsafe answer poisons the complete set. This includes metadata, private, loopback, carrier-grade NAT, and IPv4-mapped IPv6 forms.

In [ ]:
answer_sets = {
    'public': ('93.184.216.34', '2606:4700:4700::1111'),
    'mixed': ('93.184.216.34', '10.0.0.8'),
    'aws_metadata_v6': ('fd00:ec2::254',),
    'mapped_loopback': ('::ffff:127.0.0.1',),
}
{name: lab.validate_addresses(values, policy)[1] for name, values in answer_sets.items()}

## 4. Follow one admitted connection

The executor rechecks current identity, policy, and DNS, claims one grant, connects to the vetted IP, preserves the hostname as TLS SNI, and verifies the peer before releasing content.

In [ ]:
scenario = lab.build_scenario()
safe = scenario.application.fetch(attestation_id='attest:north', url=proposal, request_id='nb:safe', now=lab.NOW)
{
    'state': safe.status.value,
    'reason': safe.reason,
    'transport_call': scenario.transport.dispatches[0],
    'trust_label': safe.result.trust_label,
    'digest': safe.result.body_digest,
}

In [ ]:
receipt = asdict(safe.receipts[-1])
{
    'receipt_fields': sorted(receipt),
    'query_absent': 'query' not in receipt,
    'body_absent': 'body' not in receipt,
    'credentials_absent': not {'authorization', 'cookie', 'token'} & set(receipt),
    'trace': receipt['trace_id'],
}

## 5. Redirects re-enter admission

Automatic following is disabled. A valid redirect succeeds only because its destination independently passes URL, tenant policy, DNS, connection, TLS, and response checks.

In [ ]:
scenario = lab.build_scenario()
redirected = scenario.application.fetch(attestation_id='attest:north', url='https://research.example.test/docs/redirect', request_id='nb:redirect', now=lab.NOW)
{'state': redirected.status.value, 'hops': redirected.result.redirect_count, 'connections': scenario.transport.dispatches}

In [ ]:
scenario = lab.build_scenario()
scenario.resolver.sequences['research.example.test'] = (('93.184.216.34',), ('1.1.1.1',))
rebound = scenario.application.fetch(attestation_id='attest:north', url=proposal, request_id='nb:rotation', now=lab.NOW)
{'state': rebound.status.value, 'reason': rebound.reason, 'connections': scenario.transport.dispatches}

## 6. Constrain the response, not only the destination

Deadline, declared length, streamed wire bytes, decoded bytes, expansion ratio, media type, TLS name, and observed peer are independent controls. No failure falls back to an unrestricted client.

In [ ]:
fixtures = {
    'peer': lab.response_fixture(peer='10.0.0.8'),
    'tls': lab.response_fixture(tls_name='evil.example.test'),
    'deadline': lab.response_fixture(elapsed_ms=1501),
    'expansion': lab.response_fixture(body='x' * 5000, wire_bytes=100),
    'media': lab.response_fixture(content_type='application/octet-stream'),
}
outcomes = {}
for name, fixture in fixtures.items():
    case = lab.build_scenario()
    case.transport.fixtures[proposal] = fixture
    decision = case.application.fetch(attestation_id='attest:north', url=proposal, request_id=f'nb:{name}', now=lab.NOW)
    outcomes[name] = (decision.status.value, decision.reason, case.application.fallback_fetch_count)
outcomes

## 7. Fail closed when a dependency disappears

Resolver or transport unavailability is an error, not permission to use the host network.

In [ ]:
scenario = lab.build_scenario()
scenario.resolver.available = False
failed = scenario.application.fetch(attestation_id='attest:north', url=proposal, request_id='nb:dns-down', now=lab.NOW)
{'state': failed.status.value, 'reason': failed.reason, 'fallback_fetches': scenario.application.fallback_fetch_count, 'network_calls': scenario.transport.dispatches}

## 8. Evaluate containment and utility

The 33-case matrix has explicit populations: 5 valid, 26 attack, and 2 dependency-failure cases. The p95 is synthetic fixture time.

In [ ]:
report, observations = lab.evaluate_controls()
asdict(report)

In [ ]:
assert (report.cases, report.valid_cases, report.attack_cases, report.failure_cases) == (33, 5, 26, 2)
assert report.valid_fetch_completion_rate == 1.0
assert report.forbidden_destination_success_rate == 0.0
assert report.redirect_policy_bypass_rate == 0.0
assert report.resource_limit_enforcement_rate == 1.0
assert report.dependency_failure_error_rate == 1.0
assert report.trace_completeness_rate == 1.0
'evaluation contract satisfied'

## 9. Inspect the real SDK boundary

The OpenAI Agents SDK schema exposes only `url`. HTTPX is configured with no automatic redirects or environment proxy inheritance, but a production connection still requires a reviewed custom transport or egress proxy.

In [ ]:
sdk = load_module('course06_sdk_notebook', 'sdk_adapter.py')
sdk_proof = await sdk.sdk_boundary_demo_async()
{
    'schema_properties': sorted(sdk_proof['tool_schema']['properties']),
    'strict': sdk_proof['strict_schema'],
    'safe': sdk_proof['safe_terminal_state'],
    'private': sdk_proof['private_address_terminal_state'],
    'external_untrusted': sdk_proof['external_content_remains_untrusted'],
    'httpx': sdk_proof['httpx_controls'],
}

## Production mapping and exercise

| Teaching component | Production responsibility |
|---|---|
| In-memory workload/policy registry | authenticated workload identity and versioned policy service |
| Synthetic resolver | controlled resolver with IPv4/IPv6 answer policy and telemetry |
| One-use grant dictionary | transactional, integrity-protected grant store |
| Synthetic transport | egress proxy or custom transport binding vetted IP to connection authority |
| Response fixtures | streaming HTTP stack with cancellation, decompression, content, malware/DLP policy |
| In-memory receipts | redacted, correlated, tamper-resistant telemetry |

**Exercise:** draw the network path from an application pod through its egress gateway to the internet. Mark DNS, IPv4, IPv6, metadata, proxy failure, and direct-bypass paths. For every arrow, name the owner, control, evidence, and fail-closed behavior.

## Checkpoint and completion

A permitted hostname changes from one public address to another after grant issuance. Deny the stale grant, obtain a fresh authorization, bind the newly vetted IP to the connection, preserve the original Host/SNI, and verify the certificate and peer. Public classification alone does not prove continuity.

You are done when you can explain why URL validation, DNS filtering, a client timeout, or network policy alone is incomplete—and can show which independent layer enforces each part of the invariant.